# Entropy & Redundancy k-mer Length Selection — Revised Analysis

**TrAP (Transformers Analysis Pipeline) · LINE-1 retroelement detection**

This notebook is the analysis + visualization layer for the revised, rigorous
k-mer length selection. It addresses the peer-review concerns:

| Reviewer | Concern | Addressed by |
|---|---|---|
| **R1** | The Supp. Fig. 1 "entropy plateau" may be **sampling saturation**, not a real plateau | Bias-corrected estimators (Miller–Madow, Chao–Shen) + bootstrap CIs + **rarefaction** curves |
| **R1** | Define *entropy* and *redundancy* operationally | Methods section below (formulas + empirical distribution) |
| **R1** | Citation for the "2009" k-mer claim | Citation placeholder in Methods |
| **R2** | Move formulas to Methods; state the **selection criterion** | Methods section + quantitative 3-gate criterion |
| **R3** | Justify k **beyond** the plateau | k → classification **ablation** overlay |

**Design.** All heavy computation is done deterministically in Python
(`python -m trap.analysis run`), which writes tidy, version-controlled artifacts
(`spectrum.csv`, `ablation.csv`, `selection.json`) plus a `manifest.json`
(git commit, seed, input SHA-256). This notebook only *reads* those artifacts and
renders figures, so the figures are reproducible from the pinned inputs. The
Python ↔ R boundary keeps numerical results out of the notebook and under test.

> **Prerequisite.** Generate the artifacts first:
> ```bash
> python scripts/data/fetch_entropy_inputs.py          # remote, deterministic
> python -m trap.analysis run --config config/analysis/entropy_redundancy.yaml
> ```
> Override the input/figure directories with the `TRAP_ER_RESULTS` /
> `TRAP_ER_FIGURES` environment variables (used for testing on a small corpus).


## Methods — operational definitions and selection criterion

### Empirical k-mer distribution
For a corpus of standardized sequences (upper-case `ACTG`, non-`ACTG` removed —
the same standardization the tokenizer sees), every sequence is decomposed into
overlapping k-mers with a 1-nt sliding window. Let $c_k(w)$ be the number of
occurrences of k-mer $w$ and $N_k = \sum_w c_k(w)$ the total. The **empirical
probability** of k-mer $w$ is

$$ p_k(w) = \frac{c_k(w)}{N_k}. $$

By default k-mers are counted **forward-strand** (transcript orientation,
matching the tokenizer); a `canonical` mode collapses each k-mer with its
reverse complement (the Salmon/jellyfish convention) for strand-agnostic corpora.

### Shannon entropy (information content)
$$ H_k = -\sum_w p_k(w)\,\log_2 p_k(w) \quad \text{(bits).} $$

The plug-in (maximum-likelihood) estimator $\hat H_k$ is **negatively biased**
when the number of possible k-mers ($4^k$) approaches or exceeds $N_k$ — exactly
the regime that can manufacture an apparent plateau. We therefore also report two
bias-corrected estimators:

* **Miller–Madow:** $\;\hat H_k^{MM} = \hat H_k + \dfrac{K-1}{2 N_k \ln 2}$, where
  $K$ is the number of *observed* k-mers.
* **Chao–Shen:** a coverage-adjusted Horvitz–Thompson estimator using Good–Turing
  sample coverage $C = 1 - f_1/N_k$ ($f_1$ = number of singletons), robust when
  many k-mers are seen once.

Uncertainty is a **percentile bootstrap** 95% CI (multinomial resampling of k-mer
occurrences), seeded for reproducibility.

### Redundancy (three operational measures)
$$
R^{\text{Shannon}}_k = 1 - \frac{H_k}{\log_2 4^{k}} = 1 - \frac{H_k}{2k},\qquad
R^{\text{obs}}_k = 1 - \frac{H_k}{\log_2 K},\qquad
R^{\text{dup}}_k = 1 - \frac{K}{N_k}.
$$

$R^{\text{Shannon}}$ is Shannon's classical redundancy against the maximum
possible entropy; $R^{\text{obs}}$ is relative to the realized alphabet; and
$R^{\text{dup}}$ is the fraction of k-mer occurrences that are repeats.

### Relative entropy vs a uniform background
For a uniform background $U$ over a support of size $m$,
$$ D_{KL}(P_k \parallel U) = \log_2 m - H_k. $$
We report $m = 4^k$ (full k-mer space) and $m = K$ (observed), documenting both.

### Quantitative k-selection criterion (replaces "plateau by eye")
The recommended $k$ is the smallest $k$ satisfying three explicit gates:

1. **Saturation gate (R1).** The bias-corrected entropy at $k$ is stable as the
   corpus grows: the relative change between the ~50% and 100% subsample is below
   $\varepsilon$. A $k$ whose entropy is still climbing with $N$ is under-sampled,
   so any plateau there is a sampling artefact.
2. **Marginal-gain gate (R1/R2).** Diminishing returns begin at the first $k$
   whose per-step gain $\Delta H_k = H_k - H_{k-1}$ (bias-corrected) falls below
   $\tau$ bits.
3. **Task gate (R3).** The smallest $k$ whose downstream macro-F1 is
   statistically indistinguishable from the best (within one CV std).

The criterion is reported with every gate value and a plain-language verdict, and
is explicit about whether the diminishing-returns point lies inside the reliably
sampled regime — so it can **corroborate or revise** the manuscript's $k=16$
plateau / $k=17$ operational choice rather than assuming it.

> **Citation placeholder (R1).** The claim that systematic attention to optimal
> k-mer length emerged ~2009 should cite **Wu, Jun, Sims & Kim (2009)** on k-mer
> length selection via cumulative relative entropy — *confirm and add to
> `references.bib`.*


## 1. Setup — Libraries, Config, Paths, Palette

In [ ]:
## Reproducibility & environment capture -------------------------------
set.seed(3469)
suppressPackageStartupMessages({
  library(data.table)
  library(ggplot2)
  library(jsonlite)
})

In [ ]:
options(repr.plot.width = 18, repr.plot.height = 9)
theme_set(theme_bw(base_size = 28.125))
results_dir     <- file.path("..", "results/entropy_redundancy")
figures_dir     <- file.path("..", "reports/figures/entropy_redundancy")

dir.create(figures_dir, recursive = TRUE, showWarnings = FALSE)
cat("results_dir:", results_dir, "\n")
cat("figures_dir:", figures_dir, "\n")

# Provenance: read the Python compute manifest and assert seed agreement.
manifest_path <- file.path(results_dir, "manifest.json")
if (file.exists(manifest_path)) {
  manifest <- fromJSON(manifest_path)
  cat("\n-- compute manifest --\n")
  cat("trap_version:", manifest$trap_version, "\n")
  cat("git_commit:  ", manifest$git_commit, "\n")
  cat("timestamp:   ", manifest$timestamp_utc, "\n")
  cat("seed:        ", manifest$seed, "\n")
  if (!is.null(manifest$seed)) stopifnot(manifest$seed == 3469)
  cat("inputs:\n"); print(manifest$inputs)
} else {
  warning("manifest.json not found in results_dir — run `python -m trap.analysis run` first.")
}


In [ ]:
## Load tidy artifacts -------------------------------------------------
combined <- file.path(results_dir, "spectrum.csv")
if (file.exists(combined)) {
  spectrum <- fread(combined)
} else {
  spectrum_files <- list.files(results_dir, pattern = "\\.spectrum\\.csv$", full.names = TRUE)
  stopifnot("No spectrum CSV found — run the analysis CLI." = length(spectrum_files) > 0)
  spectrum <- rbindlist(lapply(spectrum_files, fread), use.names = TRUE, fill = TRUE)
}
cat("Loaded", nrow(spectrum), "spectrum rows;",
    "corpora:", paste(unique(spectrum$corpus), collapse = ", "), "\n")

ablation_path <- file.path(results_dir, "ablation.csv")
ablation <- if (file.exists(ablation_path)) fread(ablation_path) else NULL
if (is.null(ablation)) message("ablation.csv absent — task-gate plots will be skipped.")

selection_path <- file.path(results_dir, "selection.json")
selection <- if (file.exists(selection_path)) fromJSON(selection_path, simplifyVector = FALSE) else NULL

head(spectrum)


In [ ]:
## Validation checks (fail loudly) ------------------------------------
required_cols <- c("corpus", "k", "subsample_frac", "replicate", "n_sequences",
                   "h_mle", "h_miller_madow", "h_chao_shen",
                   "kl_uniform_full", "kl_uniform_observed",
                   "redundancy_shannon", "redundancy_observed", "redundancy_duplication",
                   "good_turing_coverage", "h_ci_low", "h_ci_high")
missing_cols <- setdiff(required_cols, names(spectrum))
stopifnot("missing expected columns" = length(missing_cols) == 0)

# Bootstrap CI brackets the point estimate.
stopifnot(all(spectrum$h_ci_low <= spectrum$h_mle + 1e-9))
stopifnot(all(spectrum$h_mle    <= spectrum$h_ci_high + 1e-9))
# Bias corrections never decrease the estimate.
stopifnot(all(spectrum$h_miller_madow >= spectrum$h_mle - 1e-9))
# Entropy can never exceed log2(4^k) = 2k bits.
stopifnot(all(spectrum$h_mle <= 2 * spectrum$k + 1e-6))
stopifnot(all(spectrum$k >= 1))

# Independent re-derivation of the uniform reference: H(uniform over n) = log2(n).
unif_H <- function(n) { p <- rep(1 / n, n); -sum(p * log2(p)) }
stopifnot(abs(unif_H(8) - 3) < 1e-9)

cat("All validation checks passed:",
    length(required_cols), "columns,",
    length(unique(spectrum$k)), "k values,",
    length(unique(spectrum$subsample_frac)), "subsample fractions,",
    max(spectrum$replicate) + 1, "replicates.\n")


In [ ]:
## Aggregate over replicates ------------------------------------------
agg <- spectrum[, .(
  h_mle = mean(h_mle), h_mm = mean(h_miller_madow), h_cs = mean(h_chao_shen),
  h_lo = mean(h_ci_low), h_hi = mean(h_ci_high),
  kl_full = mean(kl_uniform_full), kl_obs = mean(kl_uniform_observed),
  red_shannon = mean(redundancy_shannon), red_obs = mean(redundancy_observed),
  red_dup = mean(redundancy_duplication),
  coverage = mean(good_turing_coverage), n_seq = mean(n_sequences)
), by = .(corpus, k, subsample_frac)]

full <- agg[subsample_frac == max(subsample_frac)]
setorder(full, corpus, k)
full[, dH := h_mm - shift(h_mm), by = corpus]
full[, mm_gap := h_mm - h_mle]
full[, cs_gap := h_cs - h_mle]

save_fig <- function(plot, stem, width = 16, height = 8.5) {
  for (ext in c("png", "pdf")) {
    f <- file.path(figures_dir, paste0(stem, ".", ext))
    suppressMessages(ggsave(f, plot, width = width, height = height, dpi = 300, bg = "white"))
  }
  invisible(plot)
}
null_or_na <- function(x) if (is.null(x)) NA_integer_ else as.integer(x)


In [ ]:
## Supplementary Figure 1: entropy vs k ----------------------
est <- melt(full, id.vars = c("corpus", "k"),
            measure.vars = c("h_mle", "h_mm", "h_cs"),
            variable.name = "estimator", value.name = "H")

fig1 <- ggplot() +
  geom_ribbon(data = full, aes(k, ymin = h_lo, ymax = h_hi), alpha = 0.18) +
  geom_line(data = est, aes(k, H, color = estimator)) +
  geom_point(data = est, aes(k, H, color = estimator), size = 1) +
  facet_wrap(~corpus, scales = "free_y") +
  scale_color_manual(
    values = c(h_mle = "grey55", h_mm = "#1b9e77", h_cs = "#d95f02"),
    labels = c(h_mle = "Plug-in (MLE)", h_mm = "Miller-Madow", h_cs = "Chao-Shen")) +
  labs(title = "Shannon entropy vs k-mer length",
       subtitle = "Bias-corrected estimators; shaded band = 95% bootstrap CI of the plug-in",
       x = "k-mer length (k)", y = expression(H[k]~"(bits)"), color = "Estimator")
save_fig(fig1, "supp_fig1_entropy_vs_k")
fig1


In [ ]:
## Rarefaction: is the plateau real, or sampling saturation? (R1) ------
fig_sat <- ggplot(agg, aes(k, h_mm, color = factor(subsample_frac),
                           group = subsample_frac)) +
  geom_line() + geom_point(size = 0.8) +
  facet_wrap(~corpus, scales = "free_y") +
  labs(title = "Rarefaction of entropy across subsample fractions",
       subtitle = "Curves overlap => sampling-stable (real plateau); curves spread => saturation",
       x = "k-mer length (k)", y = expression(hat(H)[k]^{MM}~"(bits)"),
       color = "Subsample\nfraction")
save_fig(fig_sat, "rarefaction_entropy_by_fraction")
fig_sat


In [ ]:
options(repr.plot.width = 11, repr.plot.height = 7)
## Rarefaction vs corpus size at the largest k -------------------------
k_top <- max(agg$k)
fig_rareN <- ggplot(agg[k == k_top], aes(n_seq, h_mm, color = corpus)) +
  geom_line() + geom_point() +
  labs(title = sprintf("Entropy vs corpus size at k = %d", k_top),
       subtitle = "A rising curve at the largest k means more data still adds information",
       x = "sequences sampled (N)", y = expression(hat(H)[k]^{MM}~"(bits)"), color = "Corpus")
save_fig(fig_rareN, "rarefaction_entropy_vs_N", width = 11, height = 6.5)
fig_rareN


In [ ]:
options(repr.plot.width = 18, repr.plot.height = 7)
## Supplementary Figure 2: redundancy vs k -------------------
red <- melt(full, id.vars = c("corpus", "k"),
            measure.vars = c("red_shannon", "red_obs", "red_dup"),
            variable.name = "measure", value.name = "R")

fig2 <- ggplot(red, aes(k, R, color = measure)) +
  geom_line() + geom_point(size = 1) +
  facet_wrap(~corpus) +
  scale_color_manual(
    values = c(red_shannon = "#1f78b4", red_obs = "#33a02c", red_dup = "#e31a1c"),
    labels = c(red_shannon = "Shannon (vs 4^k)", red_obs = "Observed (vs K)",
               red_dup = "Duplication (1 - K/N)")) +
  labs(title = "Supplementary Figure 2 (revised): redundancy vs k-mer length",
       x = "k-mer length (k)", y = "Redundancy", color = "Definition")
save_fig(fig2, "supp_fig2_redundancy_vs_k")
fig2


In [ ]:
## Marginal information gain + recommended k ---------------------------
tau <- 0.05
if (!is.null(selection)) {
  g <- selection[[setdiff(names(selection), "overall")[1]]]$gates$marginal_gain_tau
  if (!is.null(g)) tau <- g
}

rec <- NULL
if (!is.null(selection)) {
  corpora <- setdiff(names(selection), "overall")
  rec <- rbindlist(lapply(corpora, function(cn)
    data.table(corpus = cn, recommended_k = null_or_na(selection[[cn]]$recommended_k))))
}

fig_gain <- ggplot(full, aes(k, dH)) +
  geom_col(fill = "grey70") +
  geom_hline(yintercept = tau, linetype = "dashed", color = "#d95f02") +
  facet_wrap(~corpus) +
  labs(title = "Marginal information gain per k-mer step",
       subtitle = sprintf("Dashed line = gain threshold tau = %.3f bits; dotted = recommended k", tau),
       x = "k-mer length (k)", y = expression(Delta*H[k]~"(bits)"))
if (!is.null(rec)) {
  fig_gain <- fig_gain +
    geom_vline(data = rec, aes(xintercept = recommended_k),
               linetype = "dotted", color = "#D62728")
}
save_fig(fig_gain, "marginal_gain_vs_k")
fig_gain


In [ ]:
## KL divergence from the uniform background ---------------------------
fig_kl <- ggplot(full, aes(k, kl_full)) +
  geom_line(color = "#6a3d9a") + geom_point(size = 1, color = "#6a3d9a") +
  facet_wrap(~corpus, scales = "free_y") +
  labs(title = "Relative entropy vs uniform background (support = 4^k)",
       x = "k-mer length (k)", y = expression(D[KL](P[k]~"||"~U)~"(bits)"))
save_fig(fig_kl, "kl_divergence_vs_k")
fig_kl


In [ ]:
## Diagnostic: where does finite-sample bias bite? ---------------------
diagm <- melt(full, id.vars = c("corpus", "k"),
              measure.vars = c("mm_gap", "cs_gap"),
              variable.name = "correction", value.name = "bits")
fig_diag <- ggplot(diagm, aes(k, bits, color = correction)) +
  geom_line() + geom_point(size = 0.8) +
  facet_wrap(~corpus, scales = "free_y") +
  scale_color_manual(values = c(mm_gap = "#1b9e77", cs_gap = "#d95f02"),
                     labels = c(mm_gap = "Miller-Madow - MLE", cs_gap = "Chao-Shen - MLE")) +
  labs(title = "Bias-correction magnitude vs k",
       subtitle = "Large gaps flag the under-sampled regime where a plug-in plateau is untrustworthy",
       x = "k-mer length (k)", y = "correction over plug-in (bits)", color = NULL)
save_fig(fig_diag, "bias_correction_magnitude_vs_k")
fig_diag


In [ ]:
## k -> classification ablation, aligned with information content (R3) -
if (!is.null(ablation)) {
  fig_abl <- ggplot(ablation, aes(k, macro_f1_mean)) +
    geom_ribbon(aes(ymin = macro_f1_mean - macro_f1_std,
                    ymax = macro_f1_mean + macro_f1_std), alpha = 0.2, fill = "#7570b3") +
    geom_line(color = "#7570b3") + geom_point(color = "#7570b3") +
    labs(title = "k -> LINE-1 classification (Reviewer 3 ablation)",
         subtitle = "Cross-validated macro-F1 of a linear k-mer-frequency classifier; band = +/- 1 CV std",
         x = "k-mer length (k)", y = "macro-F1")
  save_fig(fig_abl, "ablation_macro_f1_vs_k", width = 11, height = 6.5)
  print(fig_abl)

  # Scaled overlay: does the task agree with information content?
  norm01 <- function(x) (x - min(x)) / (max(x) - min(x) + 1e-12)
  ent <- full[, .(k, value = norm01(h_mm),
                  series = "Entropy (Miller-Madow, scaled)"), by = corpus]
  f1 <- ablation[, .(k, value = norm01(macro_f1_mean),
                     series = "Classification macro-F1 (scaled)")]
  f1 <- rbindlist(lapply(unique(full$corpus), function(cn) cbind(corpus = cn, f1)))
  combo <- rbind(ent, f1)
  fig_combo <- ggplot(combo, aes(k, value, color = series)) +
    geom_line() + geom_point(size = 0.8) +
    facet_wrap(~corpus) +
    labs(title = "Information content vs downstream task performance",
         subtitle = "Both min-max scaled; agreement supports the selected k",
         x = "k-mer length (k)", y = "scaled value", color = NULL) +
    theme(legend.position = "bottom")
  save_fig(fig_combo, "entropy_vs_task_overlay")
  print(fig_combo)
} else {
  message("No ablation table; skipping task-gate figures.")
}


In [ ]:
## Comparative analysis & selection verdict ---------------------------
if (!is.null(selection)) {
  for (cn in setdiff(names(selection), "overall")) {
    s <- selection[[cn]]
    cat(sprintf("\n=== %s ===\n", cn))
    cat(sprintf("  recommended k            : %s\n", s$recommended_k))
    cat(sprintf("  saturation-safe k (max)  : %s\n", s$saturation_safe_k_max))
    cat(sprintf("  diminishing-returns k    : %s\n", s$diminishing_returns_k))
    cat(sprintf("  within reliable regime   : %s\n", s$diminishing_returns_within_reliable_regime))
    cat(sprintf("  task-plateau k           : %s\n", s$task_plateau_k))
    cat(sprintf("  corroborates k=16/17     : %s\n", s$corroborates_manuscript_k16_17))
    cat(sprintf("  verdict: %s\n", s$verdict))
  }
} else {
  message("selection.json absent — run `python -m trap.analysis select-k`.")
}


## Assumptions & limitations

- **Strand convention.** Entropy is computed on **forward-strand** k-mers
  (transcript orientation, matching the tokenizer). The `canonical` mode
  (reverse-complement collapse) is available for strand-agnostic corpora; it
  changes the distribution and is reported as a separate run, not mixed in.
- **Finite-sample bias.** Even Miller–Madow / Chao–Shen are first-order/coverage
  corrections; for very large $k$ where $4^k \gg N$ the corpus is intrinsically
  under-sampled. The **saturation gate** is the safeguard: we only trust a
  plateau in the regime where entropy is stable as $N$ grows.
- **Sliding-window dependence.** Overlapping k-mers are not independent, so $H_k$
  measures the empirical k-mer distribution's entropy, not a per-symbol entropy
  rate; comparisons across $k$ are interpreted accordingly.
- **Corpus scope.** The analysis covers the GENCODE v48 transcriptome and the
  RepeatMasker LINE-1 corpus. The whole genome is an optional, supported third
  corpus but is not required for the k decision here.
- **Ablation is a linear proxy.** The k → classification gate uses a fast,
  deterministic logistic-regression classifier on hashed k-mer frequencies. It
  isolates the effect of k-mer *resolution* from model capacity and is **not** the
  full ALBERT pipeline; genuine ALBERT accuracies can be dropped into the same
  `ablation.csv` schema to replace the proxy.
- **Reproducibility.** Every figure derives from artifacts whose `manifest.json`
  pins the git commit, seed, and input SHA-256; re-running the Python CLI with the
  same seed regenerates byte-identical CSVs, and this notebook sets `set.seed()`
  and captures `sessionInfo()`.


In [ ]:
## Environment capture -------------------------------------------------
sessionInfo()
